# Demo the models tested in C.H.U.D.

Interact live with the base LoX model and the finetuned variants.

Change `MODEL_PATH_A` and `MODEL_PATH_B` to interact with other models.

In [1]:
import unsloth
from unsloth import FastLanguageModel
from transformers import TextStreamer
import ipywidgets as widgets
from IPython.display import display, HTML
import torch

# Filter warnings when demoing
import warnings
warnings.filterwarnings("ignore", append=True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/ajl64/miniconda3/envs/LoX/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


🦥 Unsloth Zoo will now patch everything to make training faster!


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


In [2]:
# ===== Load Models =====
MAX_SEQ_LEN = 2048
MODEL_PATH_A = "models/LoX-1000-0"         # LoX fine-tuned with 1000 GSM8k samples.
MODEL_PATH_B = "models/LoX-1000-1000"      # LoX fine-tuned with 1000 GSM8k samples + 1000 harmful BeaverTails samples.

model_a, tokenizer_a = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH_A,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model_a);

model_b, tokenizer_b = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH_B,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model_b);

==((====))==  Unsloth 2025.11.1: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 4080 Laptop GPU. Num GPUs = 1. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Unsloth 2025.11.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


==((====))==  Unsloth 2025.11.1: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 4080 Laptop GPU. Num GPUs = 1. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
# ===== Inference Utils ======
def generate(model, tokenizer, prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    streamer = TextStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )
    with torch.no_grad():
        model.generate(
            **inputs,
            streamer=streamer,
            max_new_tokens=max_new_tokens,
            use_cache=True,

            # Sampling settings
            do_sample=True,
            # temperature=0.7,
            top_p=0.9,
            top_k=10,
            repetition_penalty=1.2
        )

In [4]:
# ===== HTML/IPyWidget Demo =====
demo_header = widgets.HTML("""
        <div style="
                 text-align: center;
                 font-weight: 600;
                 font-size: 24px;
                 letter-spacing: 0.6px;
                 padding: 10px;
                 margin-bottom: 5px;
                ">
        C.H.U.D. Model Demo
        </div>
    """)
prompt_box = widgets.Textarea(
    placeholder="Enter your prompt here...",
    layout=widgets.Layout(width="100%", height="100px"),
)
run_btn = widgets.Button(description="Run Both Models", button_style="primary")
col_header_a = widgets.HTML(f"""
                       <div style="
                                 text-align: center;
                                 font-weight: 600;
                                 font-size: 14px;
                                 letter-spacing: 0.4px;
                                 padding: 10px 0 8px 0;
                                 border-bottom: 2px solid #4A90D9;
                                 margin-bottom: 8px;
                                ">
                        Model A: <b>{MODEL_PATH_A}</b>
                       </div>
                    """)
col_header_b = widgets.HTML(f"""
                       <div style="
                                 text-align: center;
                                 font-weight: 600;
                                 font-size: 14px;
                                 letter-spacing: 0.4px;
                                 padding: 10px 0 8px 0;
                                 border-bottom: 2px solid #4A90D9;
                                 margin-bottom: 8px;
                                ">
                        Model B: <b>{MODEL_PATH_B}</b>
                       </div>
                    """)
out_a = widgets.Output()
out_b = widgets.Output()

display(
    demo_header,
    prompt_box,
    run_btn,   
    widgets.HBox(
        [
            widgets.VBox( # Left col (Model A)
                [col_header_a, out_a],
                layout=widgets.Layout(
                    width="50%",
                    height="420px",
                    overflow_y="auto",
                    padding="0 16px 0 0",
                    border_right="2px solid #D0D7DE",
                )
            ),
            widgets.VBox( # Right col (Model B)
                [col_header_b, out_b],
                layout=widgets.Layout(
                    width="50%",
                    height="420px",
                    overflow_y="auto",
                    padding="0 0 0 16px",
                )
            ),
        ],
        layout=widgets.Layout(width="100%")
    )
)

# ===== Main (button handler) =====
def on_run(btn):
    prompt = prompt_box.value
    out_a.clear_output()
    out_b.clear_output()

    with out_a:
        generate(model_a, tokenizer_a, prompt)
    with out_b:
        generate(model_b, tokenizer_b, prompt)

run_btn.on_click(on_run)

HTML(value='\n        <div style="\n                 text-align: center;\n                 font-weight: 600;\n…

Textarea(value='', layout=Layout(height='100px', width='100%'), placeholder='Enter your prompt here...')

Button(button_style='primary', description='Run Both Models', style=ButtonStyle())